# Signal IC Analysis
This notebook calculates the Information Coefficient (IC) and Information Coefficient Information Ratio (ICIR) for the base signals and engineered features against the forward returns.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Set style
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#161b22", "figure.facecolor": "#0d1117"})

In [ ]:
print("Loading data/features.csv...")
df = pd.read_csv('../data/features.csv')
print(f"Loaded {len(df):,} rows.")

target_col = 'forward_return_100'
base_signals = ['microprice', 'ofi', 'vpin', 'spread_bps', 'realized_vol', 'stat_arb_zscore']

# Filter warmed up only
df = df[(df['is_warmed_up'] == 1) & (df['mid_price'] > 0)].copy()
df = df.dropna(subset=[target_col])

In [ ]:
ic_records = []
for col in base_signals:
    if col in df.columns:
        # Rank correlation
        ic, pval = spearmanr(df[col], df[target_col])
        ic_records.append({'Signal': col, 'IC': ic, 'p-value': pval})

ic_df = pd.DataFrame(ic_records).sort_values(by='IC', ascending=False)
print("Information Coefficient (IC) Table")
display(ic_df)

In [ ]:
# Plot Heatmap
corr = df[base_signals + [target_col]].corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".3f", center=0)
plt.title('Signal Rank Correlation (IC) Heatmap', color='white')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()